In [1]:
# ============================================================
# DTI-GNN V3 — LOCAL GPU SETUP
# ============================================================

import os
import gc
import pickle
import json

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from scipy.stats import pearsonr

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

import esm

# ------------------------------------------------------------
# Local project paths
# ------------------------------------------------------------

PROJECT_DIR = r"C:\DTI_GNN"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
EVAL_DIR = os.path.join(PROJECT_DIR, "evaluation")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        f"GPU memory: "
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

PyTorch version: 2.11.0+cu128
CUDA available: True
Device: cuda
GPU: NVIDIA RTX 2000 Ada Generation
GPU memory: 16.00 GB


In [2]:
# ============================================================
# LOAD V3 TARGET INFORMATION
# ============================================================

# Load target index mapping
with open(
    os.path.join(DATA_DIR, "target_to_idx.pkl"),
    "rb"
) as f:
    target_to_idx = pickle.load(f)

# Load raw protein sequences
with open(
    os.path.join(DATA_DIR, "target_sequences.pkl"),
    "rb"
) as f:
    target_sequences = pickle.load(f)

print("Number of targets:", len(target_to_idx))
print("Number of target sequences:", len(target_sequences))

# Verify that the two objects match
assert len(target_to_idx) == len(target_sequences), \
    "target_to_idx and target_sequences have different sizes!"

print("Target files loaded successfully.")

Number of targets: 2667
Number of target sequences: 2667
Target files loaded successfully.


In [3]:
# ============================================================
# LOAD PRE-BUILT V3 TRAINING DATA
# ============================================================

train_data_path = os.path.join(DATA_DIR, "train_data_v2.pt")

train_data = torch.load(
    train_data_path,
    weights_only=False
)

print("Training examples:", len(train_data))

# Check the first graph
sample = train_data[0]

print("\nFirst training sample:")
print(sample)

print("\nNode feature shape:", sample.x.shape)
print("Edge index shape:", sample.edge_index.shape)
print("Target index:", sample.target_idx)
print("Target value (pIC50):", sample.y)

Training examples: 569030

First training sample:
Data(x=[27, 18], edge_index=[2, 56], y=[1], target_idx=[1])

Node feature shape: torch.Size([27, 18])
Edge index shape: torch.Size([2, 56])
Target index: tensor([0])
Target value (pIC50): tensor([7.3979])


In [4]:
# ============================================================
# ESM-2 MODEL — V3
# ============================================================

print("Loading ESM-2 35M...")

esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()

batch_converter = alphabet.get_batch_converter()

# ------------------------------------------------------------
# Freeze the entire ESM model first
# ------------------------------------------------------------

for param in esm_model.parameters():
    param.requires_grad = False

# ------------------------------------------------------------
# Unfreeze ONLY the final transformer layer
# ------------------------------------------------------------

last_layer_idx = len(esm_model.layers) - 1

for param in esm_model.layers[last_layer_idx].parameters():
    param.requires_grad = True

# ------------------------------------------------------------
# Move ESM to GPU
# ------------------------------------------------------------

esm_model = esm_model.to(device)

# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

trainable_params = sum(
    p.numel()
    for p in esm_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in esm_model.parameters()
)

print(f"ESM layers: {len(esm_model.layers)}")
print(f"Unfrozen layer: {last_layer_idx}")
print(f"Trainable ESM parameters: {trainable_params:,}")
print(f"Total ESM parameters: {total_params:,}")
print("ESM device:", next(esm_model.parameters()).device)

Loading ESM-2 35M...
ESM layers: 12
Unfrozen layer: 11
Trainable ESM parameters: 2,771,040
Total ESM parameters: 33,501,394
ESM device: cuda:0


In [5]:
# ============================================================
# DRUG ENCODER — 3-LAYER GCN
# ============================================================

class DrugEncoder(nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels=128,
        out_channels=128
    ):
        super().__init__()

        self.conv1 = GCNConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = GCNConv(
            hidden_channels,
            hidden_channels
        )

        self.conv3 = GCNConv(
            hidden_channels,
            out_channels
        )

    def forward(self, x, edge_index, batch):

        x = F.relu(
            self.conv1(x, edge_index)
        )

        x = F.relu(
            self.conv2(x, edge_index)
        )

        x = self.conv3(
            x,
            edge_index
        )

        # Convert variable-sized molecular graphs
        # into one vector per drug
        x = global_mean_pool(
            x,
            batch
        )

        return x

In [6]:
from torch_geometric.data import Data

In [7]:
# ============================================================
# TARGET ENCODER — ESM + PROJECTION
# ============================================================

class TargetEncoder(nn.Module):

    def __init__(
        self,
        esm_model,
        batch_converter,
        target_sequences,
        hidden_dim=128,
        esm_dim=480
    ):
        super().__init__()

        self.esm_model = esm_model
        self.batch_converter = batch_converter
        self.target_sequences = target_sequences

        self.proj = nn.Sequential(
            nn.Linear(esm_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

    def forward(
        self,
        target_idx_batch,
        device
    ):

        # ----------------------------------------------------
        # Find unique targets in this batch
        # ----------------------------------------------------

        unique_idx, inverse = torch.unique(
            target_idx_batch,
            return_inverse=True
        )

        # ----------------------------------------------------
        # Get protein sequences
        # ----------------------------------------------------

        seqs = [
            (
                str(i.item()),
                self.target_sequences[i.item()]
            )
            for i in unique_idx
        ]

        # ----------------------------------------------------
        # Convert sequences to ESM tokens
        # ----------------------------------------------------

        _, _, tokens = self.batch_converter(seqs)

        tokens = tokens.to(device)

        # ----------------------------------------------------
        # Run ESM
        # ----------------------------------------------------

        results = self.esm_model(
            tokens,
            repr_layers=[12],
            return_contacts=False
        )

        reps = results["representations"][12]

        # ----------------------------------------------------
        # Mean-pool each protein representation
        # ----------------------------------------------------

        pooled = []

        for i, (_, seq) in enumerate(seqs):

            protein_repr = reps[
                i,
                1:len(seq) + 1
            ]

            pooled.append(
                protein_repr.mean(0)
            )

        pooled = torch.stack(pooled)

        # ----------------------------------------------------
        # Map unique target representations back to
        # every example in the batch
        # ----------------------------------------------------

        projected_unique = self.proj(pooled)

        # ----------------------------------------------------
        # Project 480 -> 256 -> 128
        # ----------------------------------------------------

        return projected_unique[inverse]

In [8]:
# ============================================================
# DTI MODEL V3
# ============================================================

class DTIModelV3(nn.Module):

    def __init__(
        self,
        drug_in_channels,
        esm_model,
        batch_converter,
        target_sequences,
        hidden_dim=128
    ):
        super().__init__()

        # Drug encoder
        self.drug_encoder = DrugEncoder(
            drug_in_channels,
            hidden_dim,
            hidden_dim
        )

        # Target encoder
        self.target_encoder = TargetEncoder(
            esm_model,
            batch_converter,
            target_sequences,
            hidden_dim
        )

        # Drug (128) + Target (128) = 256
        self.fusion = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(
        self,
        x,
        edge_index,
        batch,
        target_idx,
        device
    ):

        # Drug representation
        drug_vec = self.drug_encoder(
            x,
            edge_index,
            batch
        )

        # Target representation
        target_vec = self.target_encoder(
            target_idx,
            device
        )

        # Combine drug + target
        combined = torch.cat(
            [drug_vec, target_vec],
            dim=1
        )

        # Predict pIC50
        output = self.fusion(combined)

        return output.squeeze(-1)

In [9]:
# ============================================================
# V3 RANGE-WEIGHTED LOSS
# ============================================================

def compute_bin_weights(
    y_values,
    n_bins=20,
    max_weight=10.0,
    smoothing="sqrt"
):
    """
    Compute the range-weighted loss used by DTI-GNN V3.

    Rare pIC50 ranges receive larger weights.
    The weighting is sqrt-smoothed and capped at 10.
    """

    values = np.asarray(y_values)

    # --------------------------------------------------------
    # Create equally spaced pIC50 bins
    # --------------------------------------------------------

    bin_edges = np.linspace(
        values.min(),
        values.max(),
        n_bins + 1
    )

    # --------------------------------------------------------
    # Assign every training example to a bin
    # --------------------------------------------------------

    bin_idx = np.digitize(
        values,
        bin_edges[1:-1]
    )

    # --------------------------------------------------------
    # Count examples in each bin
    # --------------------------------------------------------

    bin_counts = np.bincount(
        bin_idx,
        minlength=n_bins
    )

    # Prevent division by zero
    bin_counts = np.maximum(
        bin_counts,
        1
    )

    # --------------------------------------------------------
    # Compute inverse-frequency weights
    # --------------------------------------------------------

    if smoothing == "sqrt":

        bin_weights = (
            1.0 / np.sqrt(bin_counts)
        )

    else:

        bin_weights = (
            1.0 / bin_counts
        )

    # Normalize so average weight is approximately 1
    bin_weights = (
        bin_weights /
        bin_weights.mean()
    )

    # Cap extreme weights
    bin_weights = np.clip(
        bin_weights,
        None,
        max_weight
    )

    # --------------------------------------------------------
    # Print bin information
    # --------------------------------------------------------

    print("V3 Bin ranges and weights:")
    print("-" * 60)

    for i in range(n_bins):

        lo = bin_edges[i]
        hi = bin_edges[i + 1]

        print(
            f"[{lo:.2f}, {hi:.2f}): "
            f"n={bin_counts[i]}, "
            f"weight={bin_weights[i]:.3f}"
        )

    return (
        bin_edges,
        bin_weights
    )


def weighted_mse_loss(
    pred,
    target,
    bin_edges,
    bin_weights_tensor,
    device
):
    """
    Weighted MSE used by DTI-GNN V3.
    """

    # --------------------------------------------------------
    # Determine the pIC50 bin for each target
    # --------------------------------------------------------

    bin_idx = np.digitize(
        target.detach().cpu().numpy(),
        bin_edges[1:-1]
    )

    # --------------------------------------------------------
    # Get corresponding weights
    # --------------------------------------------------------

    weights = (
        bin_weights_tensor[bin_idx]
        .to(device)
    )

    # --------------------------------------------------------
    # Squared error
    # --------------------------------------------------------

    sq_error = (
        pred - target
    ) ** 2

    # --------------------------------------------------------
    # Weighted MSE
    # --------------------------------------------------------

    return (
        weights * sq_error
    ).mean()

In [10]:
# ============================================================
# COMPUTE V3 TRAINING BIN WEIGHTS
# ============================================================

# Extract pIC50 values from the training dataset
train_y = np.array([
    data.y.item()
    for data in train_data
])

# Compute the same range weights used by original V3
bin_edges, bin_weights = compute_bin_weights(
    train_y,
    n_bins=20,
    max_weight=10.0,
    smoothing="sqrt"
)

# ------------------------------------------------------------
# Move bin weights to GPU
# ------------------------------------------------------------

bin_weights_tensor = torch.tensor(
    bin_weights,
    dtype=torch.float32,
    device=device
)

print("\nBin weights tensor device:")
print(bin_weights_tensor.device)

print("\nV3 bin weights:")
print(bin_weights)

V3 Bin ranges and weights:
------------------------------------------------------------
[3.00, 3.48): n=2247, weight=0.443
[3.48, 3.97): n=4667, weight=0.307
[3.97, 4.45): n=19664, weight=0.150
[4.45, 4.94): n=32179, weight=0.117
[4.94, 5.42): n=53826, weight=0.090
[5.42, 5.91): n=54695, weight=0.090
[5.91, 6.39): n=70637, weight=0.079
[6.39, 6.88): n=68014, weight=0.080
[6.88, 7.36): n=75496, weight=0.076
[7.36, 7.85): n=64941, weight=0.082
[7.85, 8.33): n=54866, weight=0.090
[8.33, 8.82): n=35642, weight=0.111
[8.82, 9.30): n=20752, weight=0.146
[9.30, 9.79): n=7439, weight=0.243
[9.79, 10.27): n=2689, weight=0.405
[10.27, 10.76): n=863, weight=0.714
[10.76, 11.24): n=272, weight=1.273
[11.24, 11.73): n=110, weight=2.001
[11.73, 12.21): n=26, weight=4.116
[12.21, 12.70): n=5, weight=9.386

Bin weights tensor device:
cuda:0

V3 bin weights:
[0.44276224 0.30722264 0.14967049 0.11699999 0.09046406 0.08974253
 0.07896888 0.08047722 0.07638536 0.0823593  0.08960257 0.11117089
 0.14569416 

In [11]:
# ============================================================
# TRAINING DATALOADER
# ============================================================

BATCH_SIZE = 32

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Training examples:", len(train_data))
print("Batch size:", BATCH_SIZE)
print(
    "Batches per epoch:",
    len(train_loader)
)

Training examples: 569030
Batch size: 32
Batches per epoch: 17783


In [12]:
# ============================================================
# CREATE V3 MODEL
# ============================================================

model = DTIModelV3(
    drug_in_channels=18,
    esm_model=esm_model,
    batch_converter=batch_converter,
    target_sequences=target_sequences,
    hidden_dim=128
).to(device)

# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------

optimizer = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model.parameters()
    ),
    lr=5e-4
)

print("\nModel created successfully.")

trainable_total = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total trainable parameters: "
    f"{trainable_total:,}"
)

print("Optimizer: Adam")
print("Learning rate: 5e-4")


Model created successfully.
Total trainable parameters: 3,004,513
Optimizer: Adam
Learning rate: 5e-4


In [13]:
# ============================================================
# V3 TRAINING / EVALUATION EPOCH
# ============================================================

def run_epoch_v3(
    loader,
    training=True
):
    """
    Run one complete V3 epoch.

    training=True:
        Forward + backward + optimizer update

    training=False:
        Forward only
    """

    if training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_examples = 0

    with torch.set_grad_enabled(training):

        for batch in loader:

            # ------------------------------------------------
            # Move graph batch to GPU
            # ------------------------------------------------

            batch = batch.to(device)

            # ------------------------------------------------
            # Forward pass
            # ------------------------------------------------

            pred = model(
                batch.x,
                batch.edge_index,
                batch.batch,
                batch.target_idx,
                device
            )

            # ------------------------------------------------
            # V3 weighted MSE loss
            # ------------------------------------------------

            loss = weighted_mse_loss(
                pred,
                batch.y,
                bin_edges,
                bin_weights_tensor,
                device
            )

            # ------------------------------------------------
            # Training update
            # ------------------------------------------------

            if training:

                optimizer.zero_grad()

                loss.backward()

                optimizer.step()

            # ------------------------------------------------
            # Accumulate epoch loss
            # ------------------------------------------------

            total_loss += (
                loss.item()
                * batch.num_graphs
            )

            total_examples += (
                batch.num_graphs
            )

    return (
        total_loss /
        total_examples
    )

In [14]:
# ============================================================
# V3 GPU SANITY TEST
# ============================================================

print("=" * 60)
print("RUNNING GPU SANITY TEST")
print("=" * 60)

# ------------------------------------------------------------
# 1. Check device
# ------------------------------------------------------------

print("\nDevice:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

else:

    raise RuntimeError(
        "CUDA is not available. Training must NOT be started."
    )

# ------------------------------------------------------------
# 2. Check model device
# ------------------------------------------------------------

print(
    "\nModel device:",
    next(model.parameters()).device
)

# ------------------------------------------------------------
# 3. Get one batch
# ------------------------------------------------------------

test_batch = next(
    iter(train_loader)
)

test_batch = test_batch.to(device)

print("\nTest batch loaded.")
print(
    "Number of graphs:",
    test_batch.num_graphs
)

print(
    "Node features:",
    test_batch.x.shape
)

print(
    "Edges:",
    test_batch.edge_index.shape
)

print(
    "Target indices:",
    test_batch.target_idx.shape
)

# ------------------------------------------------------------
# 4. Forward pass
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    test_pred = model(
        test_batch.x,
        test_batch.edge_index,
        test_batch.batch,
        test_batch.target_idx,
        device
    )

print("\nForward pass successful.")

print(
    "Prediction shape:",
    test_pred.shape
)

print(
    "Target shape:",
    test_batch.y.shape
)

# ------------------------------------------------------------
# 5. Check weighted loss
# ------------------------------------------------------------

test_loss = weighted_mse_loss(
    test_pred,
    test_batch.y,
    bin_edges,
    bin_weights_tensor,
    device
)

print(
    "Test batch loss:",
    test_loss.item()
)

# ------------------------------------------------------------
# 6. GPU memory usage
# ------------------------------------------------------------

allocated = (
    torch.cuda.memory_allocated() / 1024**3
)

reserved = (
    torch.cuda.memory_reserved() / 1024**3
)

print(
    f"\nGPU memory allocated: {allocated:.2f} GB"
)

print(
    f"GPU memory reserved:  {reserved:.2f} GB"
)

# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GPU SANITY TEST PASSED")
print("Training can now be started.")
print("=" * 60)

RUNNING GPU SANITY TEST

Device: cuda
GPU: NVIDIA RTX 2000 Ada Generation
GPU memory: 16.00 GB

Model device: cuda:0

Test batch loaded.
Number of graphs: 32
Node features: torch.Size([1052, 18])
Edges: torch.Size([2, 2314])
Target indices: torch.Size([32])

Forward pass successful.
Prediction shape: torch.Size([32])
Target shape: torch.Size([32])
Test batch loss: 5.233334541320801

GPU memory allocated: 0.14 GB
GPU memory reserved:  7.99 GB

GPU SANITY TEST PASSED
Training can now be started.


In [ ]:
# ============================================================
# V3 TRAINING LOOP
# ============================================================

NUM_EPOCHS = 15

CHECKPOINT_EVERY = 1

print("Starting V3 training...")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

for epoch in range(1, NUM_EPOCHS + 1):

    print(f"\n{'=' * 60}")
    print(f"Epoch {epoch}/{NUM_EPOCHS}")
    print(f"{'=' * 60}")

    train_loss = run_epoch_v3(
        train_loader,
        training=True
    )

    print(
        f"Epoch {epoch:02d} | "
        f"Weighted Train Loss: {train_loss:.4f}"
    )

    # --------------------------------------------------------
    # Save checkpoint every epoch
    # --------------------------------------------------------

    if epoch % CHECKPOINT_EVERY == 0:

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"dti_model_v3_stable_epoch{epoch}.pt"
        )

        checkpoint = {
            "epoch": epoch,
            "architecture_version": 2,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": train_loss,
            "bin_edges": bin_edges,
            "bin_weights": bin_weights,
        }

        torch.save(
            checkpoint,
            checkpoint_path
        )

        print(
            f"Checkpoint saved: {checkpoint_path}"
        )

print("\nTraining completed.")

Starting V3 training...
Epochs: 15
Checkpoint directory: C:\DTI_GNN\checkpoints

Epoch 1/15


In [ ]:
# ============================================================
# FINAL V3 PRE-TRAINING CHECK
# ============================================================

print("=" * 70)
print("DTI-GNN V3 — PRE-TRAINING CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Training configuration
# ------------------------------------------------------------

NUM_EPOCHS = 15
CHECKPOINT_EVERY = 1

# ------------------------------------------------------------
# 1. CUDA
# ------------------------------------------------------------

assert torch.cuda.is_available(), \
    "CUDA is not available!"

print("\n[1] CUDA")
print("  Device:", device)
print("  GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 2. Dataset
# ------------------------------------------------------------

assert len(train_data) > 0, \
    "Training dataset is empty!"

print("\n[2] DATASET")
print("  Training examples:", len(train_data))
print("  Batch size:", BATCH_SIZE)
print("  Batches/epoch:", len(train_loader))

# ------------------------------------------------------------
# 3. Target information
# ------------------------------------------------------------

assert len(target_sequences) == len(target_to_idx), \
    "Target mapping and sequences do not match!"

print("\n[3] TARGET DATA")
print("  Targets:", len(target_sequences))

# ------------------------------------------------------------
# 4. ESM
# ------------------------------------------------------------

assert next(esm_model.parameters()).is_cuda, \
    "ESM is not on CUDA!"

esm_trainable = sum(
    p.numel()
    for p in esm_model.parameters()
    if p.requires_grad
)

print("\n[4] ESM")
print("  Device:", next(esm_model.parameters()).device)
print("  Trainable parameters:", f"{esm_trainable:,}")

# ------------------------------------------------------------
# 5. Model
# ------------------------------------------------------------

assert next(model.parameters()).is_cuda, \
    "DTI model is not on CUDA!"

model_trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("\n[5] MODEL")
print("  Device:", next(model.parameters()).device)
print("  Trainable parameters:", f"{model_trainable:,}")

# ------------------------------------------------------------
# 6. Loss weights
# ------------------------------------------------------------

assert len(bin_weights) == 20, \
    "Expected 20 pIC50 bins!"

assert len(bin_edges) == 21, \
    "Expected 21 bin edges!"

print("\n[6] WEIGHTED LOSS")
print("  Number of bins:", len(bin_weights))
print("  Maximum weight:", bin_weights.max())
print("  Minimum weight:", bin_weights.min())

# ------------------------------------------------------------
# 7. Checkpoint directory
# ------------------------------------------------------------

assert os.path.isdir(CHECKPOINT_DIR), \
    "Checkpoint directory does not exist!"

print("\n[7] CHECKPOINTS")
print("  Directory:", CHECKPOINT_DIR)

# ------------------------------------------------------------
# 8. GPU memory
# ------------------------------------------------------------

allocated = (
    torch.cuda.memory_allocated() / 1024**3
)

reserved = (
    torch.cuda.memory_reserved() / 1024**3
)

print("\n[8] GPU MEMORY")
print(f"  Allocated: {allocated:.2f} GB")
print(f"  Reserved:  {reserved:.2f} GB")

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALL PRE-TRAINING CHECKS PASSED")
print("=" * 70)

print(
    "\nTraining configuration:"
)

print(
    "  Epochs:",
    NUM_EPOCHS
)

print(
    "  Checkpoint frequency:",
    CHECKPOINT_EVERY,
    "epoch(s)"
)

print(
    "\nThe model is ready for the training-batch test."
)